# Prognosticering af Månedlige Bilforsikringsskader med PROC FORECAST

## Resumé

Et aktuarteam for hensættelser har brug for et 12-måneders fremadrettet billede af månedlige bilforsikringsskader, der viser stabil porteføljevækst plus et markant vinterløft. Denne notesbog genererer fem års syntetiske månedlige skadesantal (60 måneder, jan. 2020 - dec. 2024, fra ca. 1.460 til 2.780 skader), og bruger derefter **PROC FORECAST** til at tilpasse en trinvis autoregressiv basislinje og en multiplikativ Holt-Winters-sæsonmodel. Hver model giver 12 månedlige punktprognoser med 95 % konfidensgrænser til brug for kapacitets- og hensættelsesplanlægning. Den sæsonbestemte Holt-Winters-model forudsiger den stærkeste efterspørgsel én til to måneder frem (ca. 2.780-2.900 skader), som aftager til et lavpunkt omkring trin ni (ca. 2.200), mens den autoregressive basislinje forudsiger et jævnere fald; begge konfidensbånd udvides med horisonten.

## Datakilder

| Datasæt | Rækker | Granularitet | Nøglevariable | Beskrivelse |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | Én række pr. kalendermåned (jan. 2020 - dec. 2024) | `date` (SAS-dato, `MONYY7.`), `claim_count` (numerisk) | Syntetiske månedlige bilforsikringsskader opbygget af en lineær vækst-tendens (~9 skader/måned), en sinusformet årlig cyklus, et additivt vinterløft i dec./jan./feb., og gaussisk støj (`rand('normal')`). Seed `20240531` gør det reproducerbart. |

# Prognosticering af Månedlige Bilforsikringsskader

Teams for hensættelser og kapacitetsplanlægning hos et privatkundeforsikringsselskab har brug for et fremadrettet billede af, hvor mange **bilskader** der vil blive anmeldt hver måned. Skadesvolumen i denne portefølje vokser støt, efterhånden som porteføljen udvides, og den stiger hver vinter, når is, sne og mindre dagslys øger antallet af kollisions- og glasskader.

Denne notesbog gennemgår en komplet `PROC FORECAST`-arbejdsgang:

1. Generer en realistisk, fuldt syntetisk månedlig skadesantalsserie.
2. Tilpas en **trinvis autoregressiv (STEPAR)** basislinje, der fanger tendens plus autokorrelation.
3. Tilpas en **multiplikativ Holt-Winters (WINTERS)** model, der eksplicit modellerer den 12-måneders sæsoncyklus.
4. Sammenlign de to modeller, og fortolk den fremadrettede prognose og konfidensbånd.

Alt køres på indlejrede syntetiske data — ingen eksterne filer eller netværksadgang.

## Trin 1 — Generer den syntetiske skadesserie

DATA-trinnet nedenfor opbygger **60 månedlige observationer** (jan. 2020 til dec. 2024). For hver måned kombinerer vi fire komponenter, der afspejler en rigtig bilforsikringsportefølje:

- **Tendens** — en basislinje på ~1.800 skader, der vokser med ~9 pr. måned, efterhånden som eksponeringen stiger.
- **Årlig cyklus** — et sinus-/cosinusled, der giver en jævn sæsonbølge.
- **Vinterløft** — et additivt hop i december, januar og februar.
- **Støj** — `rand('normal', 0, 90)` for realistisk variation fra måned til måned.

`call streaminit()` fastlåser den tilfældige talstrøm, så notesbogen er reproducerbar. Variablen `date` er en ægte SAS-dato bygget med `INTNX` og formateret `MONYY7.`, og sætningen `ID date INTERVAL=MONTH` navngiver den som tidsidentifikator. De første 14 rækker vist nedenfor ligger mellem ca. 1.460 og 2.450 skader.

In [1]:
data claims;
    call streaminit(20240531);
    gør i = 0 til 59;
        /* Byg en ægte SAS-månedsdato, så INTERVAL=MONTH stemmer overens */
        date = intnx('month', '01JAN2020'd, i);
        format date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = jan ... 12 = dec */

        trend  = 1800 + 9 * i;               /* voksende eksponeringsbasis */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx in (12, 1, 2));   /* is/sne-løft */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        hvis claim_count < 0 så claim_count = 0;
        uddata;
    slut;
    behold date claim_count;
kør;

procedure udskriv data=claims(obs=14) mærkat;
    mærkat date = 'Måned' claim_count = 'Anmeldte skader';
    titel "De første 14 måneder af syntetiske bilforsikringsskader";
kør;

                                De første 14 måneder af syntetiske bilforsikringsskader                                 

  Obs   Måned  Anmeldte skader
    1   21915             2178
    2   21946             2281
    3   21975             2252
    4   22006             1974
    5   22036             2067
    6   22067             1805
    7   22097             1697
    8   22128             1619
    9   22159             1462
   10   22189             1688
   11   22220             1713
   12   22250             2008
   13   22281             2116
   14   22312             2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Trin 2 — Trinvis autoregressiv basislinje (METHOD=STEPAR)

`METHOD=STEPAR` er standarden. Den tilpasser først en tidstendensmodel — her `TREND=2` for en lineær tendens — og anvender derefter en **trinvis autoregression** på residualerne, hvor AR-lags optages og bevares efter signifikans. `AR=4` sætter et loft på fire lags for den kandiderende autoregressive orden, hvilket er rigeligt for en månedlig serie med kortsigtet momentum.

Centrale valgmuligheder brugt her:

- `LEAD=12` — prognosticér 12 måneder ud over dataene.
- `ALPHA=0.05` — 95 % konfidensgrænser.
- `OUTFULL` — stabl de historiske `ACTUAL`-rækker sammen med prognosehorisontens rækker i `OUT=`-datasættet (i stedet for kun prognoserækkerne).
- `OUTLIMIT` — tilføj de nedre/øvre konfidensgrænse-**kolonner** (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` — navngiver tidsidentifikatoren og erklærer, at serien er månedlig.

In [2]:
procedure forecast data=claims
        out=fc_stepar
        method=stepar trend=2 ar=4
        lead=12 alpha=0.05
        outfull outlimit;
    id date interval=month;
    var claim_count;
kør;

procedure udskriv data=fc_stepar(obs=10) mærkat;
    titel "STEPAR-output: Faktiske, tilpassede og prognoserækker";
kør;

                                De første 14 måneder af syntetiske bilforsikringsskader                                 

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                 STEPAR-output: Faktiske, tilpassede og prognoserækker                                  

  Obs   DATE  _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT
    1  21915  ACTUAL  2121.816667                .                .
    2  21946  ACTUAL  2167.711419                .                .
    3  21975  ACTUAL  2254.781177                .                .
    4  22006  ACTUAL  2228.505912                .                .
    5  22036  ACTUAL  1978.152296                .                .
    6  22067  ACTUAL  2030.606563                .                .
    7  22097  ACTUAL  1864.520418                .                .
    8  22128  ACTUAL  1784.855682                .                .
    9  22159  ACTUAL  1740.781553  


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### Aflæsning af OUT=-datasættet

`OUT=`-datasættet stabler rækker efter en `_TYPE_`-kolonne og bærer konfidensgrænserne som side-**kolonner**:

| Element | Type | Betydning |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | række | Den observerede `claim_count` for hver af de 60 historiske måneder. |
| `_TYPE_ = 'FORECAST'` | række | De 12 punktforudsigelser for prognosehorisonten. |
| `L95_claim_count` / `U95_claim_count` | kolonne | Nedre/øvre 95 % konfidensgrænser, udfyldt på `FORECAST`-rækkerne (mangler på `ACTUAL`-rækkerne). Det numeriske niveau afspejler `ALPHA=`. |

Så det udskrevne `OUT=` her indeholder 72 rækker: 60 `ACTUAL`-historikrækker efterfulgt af 12 `FORECAST`-horisontrækker. De første ti rækker vist ovenfor er alle `ACTUAL`, hvor konfidensgrænse-kolonnerne mangler, fordi grænserne kun knytter sig til prognoserækkerne.

> **Motornote.** SAS' `OUTFULL` skriver også en et-skridt-frem `FORECAST`-række inden for stikprøven for hver historisk måned, og `OUTRESID` tilføjer `_TYPE_='RESIDUAL'`-rækker. Jenners PROC FORECAST udsender endnu ikke disse in-sample tilpassede/residual-rækker (registreret som gap-test `400979`), så denne notesbog læser kun `ACTUAL`-historikken og den fremadrettede `FORECAST`-horisont.

## Trin 3 — Sæsonbestemt Holt-Winters-model (METHOD=WINTERS)

STEPAR-modellen fanger tendens og kortsigtet autokorrelation, men modellerer ikke eksplicit det tilbagevendende løft i december-februar. For en serie med en klar årlig rytme er **multiplikativ Holt-Winters** det bedre værktøj.

`METHOD=WINTERS` dekomponerer serien i niveau, lineær tendens (`TREND=2`) og en **multiplikativ sæsonfaktor**. `SEASONS=12` erklærer en 12-perioders (månedlig) sæsoncyklus. Vi anmoder igen om en 12-måneders `LEAD` med 95 % grænser og `OUTFULL OUTLIMIT`, så outputtet er på linje med STEPAR-kørslen.

Fordi motoren fremskriver prognosens `ID` med én enhed pr. trin (i stedet for at overholde `INTERVAL=MONTH` for horisontens datoer — gap-test `400979`), gennemgår cellen nedenfor prognosen efter **måneder frem (trin 1-12)** i stedet for at basere sig på de udskrevne kalenderdatoer.

In [3]:
procedure forecast data=claims
        out=fc_winters
        method=winters seasons=12 trend=2
        lead=12 alpha=0.05
        outfull outlimit;
    id date interval=month;
    var claim_count;
kør;

/* Behold den 12-måneders fremadrettede horisont, indekseret efter trin (1..12). */
data horizon;
    sæt fc_winters;
    behold_værdi months_ahead 0;
    hvis _type_ = 'FORECAST';
    months_ahead + 1;
    behold months_ahead claim_count l95_claim_count u95_claim_count;
kør;

procedure udskriv data=horizon mærkat;
    mærkat months_ahead     = 'Måneder frem'
          claim_count       = 'Prognosticerede skader'
          l95_claim_count   = 'Nedre 95 %'
          u95_claim_count   = 'Øvre 95 %';
    titel "Holt-Winters 12-måneders fremadrettet prognose (efter trin)";
kør;

                                 STEPAR-output: Faktiske, tilpassede og prognoserækker                                  

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                              Holt-Winters 12-måneders fremadrettet prognose (efter trin)                               

  Obs   Måneder frem  Prognosticerede skader   Nedre 95 %    Øvre 95 %
    1              1              2783.07951  2577.844742  2988.314278
    2              2             2897.355557  2607.109764  3187.601349
    3              3             2805.272075  2449.795029   3160.74912
    4              4             2664.498049  2254.028514  3074.967585
    5              5             2628.810136  2169.891244  3087.729029
    6              6              2548.39319  2045.672732  3051.113649
    7              7             2391.649756    1848.6496  2934.649912
    8              8             2239.948352  1659.456768  2820.439936
    9   


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Trin 4 — Sammenlign de to modeller direkte

Den klareste måde at vurdere, om sæsonmodellen gør gavn, er at lægge dens 12-måneders prognose ved siden af STEPAR-basislinjen, trin for trin. Vi udtrækker `FORECAST`-rækkerne fra begge `OUT=`-datasæt, indekserer hver efter måneder-frem, og sammenfletter dem, så afvigelsen er synlig med det samme.

De to metoder er enige om det brede niveau, men uenige om **formen**: Holt-Winters bærer den årlige rytme videre ind i horisonten (et højere niveau tidligt i horisonten, der aftager til et lavpunkt midtvejs og stiger igen), mens STEPAR — som kun modellerer sæsonudsving indirekte gennem AR-lags — aftager mere jævnt mod seriens gennemsnit. Forskellen mellem dem ved hvert trin er det sæsonsignal, STEPAR ikke fanger.

> SAS ville normalt drive dette tilstrækkelighedstjek med `OUTRESID`-etskridt-frem-residualer (`_TYPE_='RESIDUAL'`). Jenner udfylder endnu ikke disse rækker (gap-test `400979`), så vi sammenligner i stedet de to fremadrettede prognoser direkte — en diagnostik, der kun bruger output, motoren rent faktisk producerer.

In [4]:
/* STEPAR-horisont, indekseret efter måneder-frem. */
data stepar_h;
    sæt fc_stepar;
    behold_værdi months_ahead 0;
    hvis _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    behold months_ahead stepar;
kør;

/* WINTERS-horisont, indekseret efter måneder-frem. */
data winters_h;
    sæt fc_winters;
    behold_værdi months_ahead 0;
    hvis _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    behold months_ahead winters;
kør;

/* Sammenflet de to, og vis det sæsonudsving STEPAR misser. */
data compare;
    sammenflet stepar_h winters_h;
    efter months_ahead;
    seasonal_gap = winters - stepar;
kør;

procedure udskriv data=compare mærkat;
    mærkat months_ahead = 'Måneder frem'
          stepar        = 'STEPAR-prognose'
          winters       = 'Winters-prognose'
          seasonal_gap  = 'Winters - STEPAR';
    format stepar winters seasonal_gap 8.0;
    titel "STEPAR vs. Holt-Winters: Sammenligning af 12-måneders prognose";
kør;

                             STEPAR vs. Holt-Winters: Sammenligning af 12-måneders prognose                             

  Obs   Måneder frem  STEPAR-prognose  Winters-prognose  Winters - STEPAR
    1              1             2619              2783               164
    2              2             2537              2897               361
    3              3             2384              2805               421
    4              4             2214              2664               450
    5              5             2089              2629               540
    6              6             2010              2548               539
    7              7             1982              2392               410
    8              8             1995              2240               245
    9              9             2031              2197               166
   10             10             2075              2354               280
   11             11             2115              2423         


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Trin 5 — Prognosticér alle forretningslinjer på én gang (BY-behandling)

Rigtige hensættelseskørsler dækker flere produkter. Når dataene er sorteret efter `line_of_business`, får en `BY`-sætning `PROC FORECAST` til at tilpasse en **uafhængig sæsonmodel for hver gruppe** i ét kald. Her syntetiserer vi en Bil-portefølje (højere basisvolumen) og en Bolig-portefølje (lavere basis) og prognosticerer begge seks måneder frem. `OUTALL` skriver hele rækkesættet — `ACTUAL`-historikken og `FORECAST`-horisonten — sammen med grænsekolonnerne for hver gruppe, og vi beholder kun `FORECAST`-rækkerne til tabellen nedenfor.

In [5]:
data multi_book;
    call streaminit(20240531);
    længde line_of_business $6;
    gør lob = 1 til 2;
        hvis lob = 1 så line_of_business = 'Bil';
        ellers            line_of_business = 'Bolig';
        gør i = 0 til 47;
            date = intnx('month', '01JAN2021'd, i);
            format date monyy7.;
            mi   = mod(i, 12) + 1;
            base = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(base + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi in (12, 1, 2))
                + rand('normal', 0, 70));
            uddata;
        slut;
    slut;
    behold line_of_business date claim_count;
kør;

procedure sorter data=multi_book;
    efter line_of_business date;
kør;

procedure forecast data=multi_book
        out=fc_by
        method=winters seasons=12 trend=2
        lead=6 alpha=0.05
        outall;
    efter line_of_business;
    id date interval=month;
    var claim_count;
kør;

procedure udskriv data=fc_by(obs=12) mærkat;
    mærkat line_of_business="Forretningslinje" date="Måned" claim_count="Skader";
    hvor _type_ = 'FORECAST';
    titel "Prognoser for 6 måneder pr. forretningslinje";
kør;

                             STEPAR vs. Holt-Winters: Sammenligning af 12-måneders prognose                             


BY Group: line_of_business=Bil

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Bolig

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                      Prognoser for 6 måneder pr. forretningslinje                                      

  Obs  Forretningslinje   Måned    _TYPE_       Skader  L95_CLAIM_COUNT  U95_CLAIM_COUNT  RESIDUAL_CLAIM_COUNT
    1  Bil                23742  FORECAST  2524.596717      2359.095846      2690.097589                     .
    2  Bil                23773  FORECAST  2604.418724      2370.365147        2838.4723                     .
    3  Bil                23801  FORECAST  2576.092313      2289.436395       2862.74823                     .
    4  Bil                2383


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Bil
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Bolig
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Fortolkning af resultaterne

**Hvad modellerne fortæller hensættelsesteamet:**

- **STEPAR** genskaber den opadgående drift og det kortsigtede momentum, men dens 12-måneders prognose aftager jævnt fra ca. 2.620 (trin 1) mod ca. 1.980 midtvejs, før den driver tilbage til ca. 2.140 — den genskaber ikke en skarp vintertop, fordi den kun behandler sæsonudsving gennem autoregressive lags. Den er en fornuftig hurtig basislinje.
- **WINTERS** med `SEASONS=12` bærer den årlige rytme direkte videre gennem sine multiplikative sæsonfaktorer: dens prognose er højest tidligt i horisonten (ca. 2.780 ved trin 1, ca. 2.900 ved trin 2), aftager til et lavpunkt nær trin 9 (ca. 2.200), og stiger igen ved trin 12 (ca. 2.800). Sammenlignet med STEPAR-basislinjen ligger den **150-660 skader højere** gennem det meste af horisonten (se kolonnen `Winters - STEPAR` i trin 4) — den forskel er det sæsonsignal, den autoregressive model ikke fanger.
- Det **95 % konfidensbånd** (`L95_*`/`U95_*`-kolonnerne, styret af `ALPHA=`) udvides, efterhånden som horisonten forlænges — for WINTERS-modellen fra en bredde på ca. 410 skader ved trin 1 til ca. 1.420 ved trin 12 — det ærlige signal om, at sene horisontestimater bærer mere usikkerhed end kortsigtede. Aktuarerne bør afsætte kapital mod den øvre grænse, ikke kun punktprognosen.
- **BY-behandling** producerer Bil- og Bolig-prognoserne i én omgang, hver med sin egen sæsontilpasning. Bil-porteføljen prognosticerer i intervallet ca. 2.510-2.600, mens Bolig-porteføljen ligger nær 1.870-2.030, så hver linje beholder sit eget niveau og sæsonform — det mønster, teamet ville udvide til hvert produkt i porteføljen.

**Konklusion:** for en skadesantalsserie med en klar årlig cyklus er `METHOD=WINTERS SEASONS=12` det idiomatiske valg; STEPAR-basislinjen er et nyttigt sanity-tjek, og `OUTFULL`/`OUTLIMIT` plus en trinvis modelsammenligning lader aktuaren dimensionere den fremadrettede hensættelse med dens usikkerhedsbånd.

> **Motorpræcisionsnote.** Denne notesbog dokumenterer to nuværende begrænsninger i Jenners PROC FORECAST (gap-test `400979`): prognosehorisontens `ID` fremskrives med én enhed pr. trin i stedet for med `INTERVAL=MONTH`, så de udskrevne prognosedatoer ikke er de tilsigtede 2025-kalendermåneder (vi gennemgår i stedet horisonten efter trin); og `OUTRESID`/`OUTALL` udfylder endnu ikke de etskridt-frem `_TYPE_='RESIDUAL'`-rækker, så residualdiagnostik er erstattet af en direkte to-modelsammenligning. Selve punktprognoserne og konfidensgrænserne produceres og er det, narrativet ovenfor er baseret på.